## Data Preparation: Sell Out e Inventario

**Objetivo**. Preparar los datasets de ventas e inventario para su posterior análisis.

A partir de los hallazgos obtenidos durante el Data Proffiling, en este notebook se realizarán procesos de:

- Estandarización de tipos de datos
- Conversión y validación de fechas
- Tratamiento controlado de valores faltantes
- Validación de variables críticas
- Preparación de número de parte como campo de integración
- Integración Ventas + Inventario
- Generación de un dataset analítico

Las transformaciones realizadas estarán basadas en reglas identifciadas previamente durante el análisis de calidad de datos

##Validación de Fechas

In [0]:
#Librerías
import numpy as np #cálculos y condiciones
import pandas as pd # Trabajar con DataFrames
from pathlib import Path # Para trabajar con rutas de archivos de manera más organizada


# Contrucción de rutas de datasets y validación de archivos exists
RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path("../data/processed")

# Lectura de dataset
sales = RAW_PATH / 'Sales_dataset.csv'
print("Sales file exists:", sales.exists())

inventory = RAW_PATH / 'Inventory_dataset.csv'
print("Inventory file exists:", inventory.exists())

#Crear carpeta de salida
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Carga de datos
dfSales = pd.read_csv(sales, dtype={
    "cd_NumParte": "string",
    "cd_IdClienteFC": "string"
    }
)

dfInventory = pd.read_csv(inventory, dtype= {"cd_NumParte": "string"})

In [0]:
#Confirmar carga
print(
    f"Sales: {dfSales.shape[0]} filas, "
    f"{dfSales.shape[1]} columnas"
)
print(
    f"Inventory: {dfInventory.shape[0]} filas, "
    f"{dfInventory.shape[1]} columnas"
)


In [0]:
# Estandarizar el número de parte
## Evitar esapcios innecesarios, actúa como Trim, evitando errores en el Merge
dfSales["cd_NumParte"] = (
    dfSales["cd_NumParte"]
    .str.strip()
)

dfInventory["cd_NumParte"] = (
    dfInventory["cd_NumParte"]
    .str.strip()
)

In [0]:
# Convertir fechas
## qué signifcia "coerce", si Pandas encuentra una fecha en formato incorrecto, en lugar de detener el proceso, convierte la fecha en NaT (Not a Time)
dfSales["fh_Fecha"] = pd.to_datetime(
    dfSales["fh_Fecha"],
    errors="coerce",
    dayfirst=True
)

dfInventory["fh_FechaCarga"] = pd.to_datetime(
    dfInventory["fh_FechaCarga"],
    errors="coerce",
    dayfirst=True
)

In [0]:
#Comprobando fechas inválidas
invalid_sales_dates = (
    dfSales["fh_Fecha"]
    .isna()
    .sum()
)

invalid_inventory_dates = (
    dfInventory["fh_FechaCarga"]
    .isna()
    .sum()
)

print(
    "Fechas inválidas en Sales:",
    invalid_sales_dates
)

print(
    "Fechas inválidas en Inventory:",
    invalid_inventory_dates
)

**Pipeline de Calidad**

dato RAW

   ↓
to_datetime()

   ↓
errors="coerce"

   ↓
NaT

   ↓
isna()

   ↓
conteo


##Variables críticas

In [0]:
# Qué variables necesitamos
sales_required = [
    "cd_NumParte",
    "fh_Fecha",
    "ct_Cantidad",
    "im_PrecioVenta"
]

inventory_required = [
    "cd_NumParte",
    "ct_Existencia"
]


In [0]:
#Nulos críticos en campos identificados como críticos de cada dataset
print("Nulos críticos en Sales:")

display(
    dfSales[sales_required]
    .isna()
    .sum()
)

print("Nulos críticos en Inventory:")

display(
    dfInventory[inventory_required]
    .isna()
    .sum()
)


In [0]:
# Identifciando campos cantidades válidas
dfSales["fl_CantidadValida"] = (
    dfSales["ct_Cantidad"] > 0
)

display(
    dfSales[
        [
            "cd_NumParte",
            "ct_Cantidad",
            "fl_CantidadValida"
        ]
    ].head(20)
)

##Resumen de estandarización

Durante esta etapa se realizaron las siguientes acciones:

- Los números de parte se conservaron como variables de texto
- Se eliminaron espacios innecesarios de los indicadores
- Los identificadores de cliente se concervan como texto
- Las fechsa fueron convertidas a un formato de fecha válido
- Los valores de fecha que no pudieron convertirse fueron identifciados para revisión
- Se validaron campos críticos para ventas e inventario
- Las cantidades menores o iguales a cero no fueron eliminadas automaticamente; se identificaron mediante una bandera de calidad.

Estas reglas permiten conservar la trazabilidad de los datos originales y preparar ambos datasets para su integración.


## Cleaning Rules

Al terminar tendremos:
- Sales_clean
- Inventory_clean
- Registros originales conservados
- Registros problemáticos marcados, no eliminados arbitrariamente
- Un pequeño resumen de calidad después de la preparación.

In [0]:
#Crear copias de trabajo
##Numca transformaremos directamente los DataFrames originales

sales_clean = dfSales.copy()
inventory_clean = dfInventory.copy()

In [0]:
##Normalizar números de parte
## Ya se eliminaron espacios pero también conviene homologar mayúsculas y minúsculas

sales_clean["cd_numParte"] = (
    sales_clean["cd_NumParte"]
    .str.strip()
    .str.upper()
)

inventory_clean["cd_numParte"] = (
    inventory_clean["cd_NumParte"]
    .str.strip()
    .str.upper()
)



In [0]:
#Identificar números de parte faltantes

sales_clean["fl_ParteValida"] = (
    sales_clean["cd_NumParte"].notna()
    &
    (sales_clean["cd_NumParte"] != "")
)

inventory_clean["fl_ParteValida"] = (
    inventory_clean["cd_NumParte"].notna()
    &
    (inventory_clean["cd_NumParte"] != "")
)

## Aquí ~, significa NOT
print(
    "Sales - partes inválidas:",
    (~sales_clean["fl_ParteValida"]).sum()
)

print(
    "Inventory - partes inválidas:",
    (~inventory_clean["fl_ParteValida"]).sum()
)

In [0]:
#Regla de cantidad

##De Profiling ya sabemos que una cantidad <= 0 merece revisión
sales_clean["fl_CantidadValida"] = (
    sales_clean["ct_Cantidad"].notna()
    &
    (sales_clean["ct_Cantidad"] > 0)
)

display(sales_clean[
    [
        "cd_NumParte",
        "ct_Cantidad",
        "fl_CantidadValida"
    ]
].head(20))

In [0]:
#Validar precio
sales_clean["fl_PrecioValido"] = (
    sales_clean["im_PrecioVenta"].notna()
    &
    (sales_clean["im_PrecioVenta"] > 0)
)

print(
    "Precios inválidos:",
    (~sales_clean["fl_PrecioValido"]).sum()
)

In [0]:
# Validar Fechas
sales_clean["fl_FechaValida"] = (
    sales_clean["fh_Fecha"].notna()
)

inventory_clean["fl_FechaValida"] = (
    inventory_clean["fh_FechaCarga"].notna()
)

#Crear una Regla General de Calidad

Una venta estará habilitada si:
Parte válida
AND
Cantidad válida
AND
Precio válido
AND
Fecha válida

In [0]:
#Regla en Python
sales_clean["fl_RegistroValido"] = (
    sales_clean["fl_ParteValida"]
    &
    sales_clean["fl_CantidadValida"]
    &
    sales_clean["fl_PrecioValido"]
    &
    sales_clean["fl_FechaValida"]
)

print(
    sales_clean["fl_RegistroValido"]
    .value_counts()
)

#Calculando el Porcentaje
valid_percentage = (
    sales_clean["fl_RegistroValido"]
    .mean() #Promedio
    * 100
)

print(
    f"Registros válidos: "
    f"{valid_percentage:.2f}%"
)

#En el caso de que una condición booleana sea False, se convierte en proporción, ejemplo (1+1+1+0)/4=0.75, es decir 75% es válido

In [0]:
#Crear una columna con el motivo
def quality_issue(row):

    issues = []

    if not row["fl_ParteValida"]:
        issues.append("Número de parte inválido")

    if not row["fl_CantidadValida"]:
        issues.append("Cantidad inválida")

    if not row["fl_PrecioValido"]:
        issues.append("Precio inválido")

    if not row["fl_FechaValida"]:
        issues.append("Fecha inválida")

    if len(issues) == 0:
        return "OK"

    return " | ".join(issues)

In [0]:
sales_clean["tx_QualityIssue"] = (
    sales_clean.apply(
        quality_issue,
        axis=1
    )
)

In [0]:
#Revisión de inventario
display(
    sales_clean[
        [
            "cd_NumParte",
            "ct_Cantidad",
            "im_PrecioVenta",
            "fl_RegistroValido",
            "tx_QualityIssue"
        ]
    ].head(20)
)

In [0]:
#Crear el dataset válido para análisis, sin perder el original
sales_analysis = (
    sales_clean[
        sales_clean["fl_RegistroValido"]
    ]
    .copy()
)

#Comparamos
print(
    "Registros originales:",
    len(sales_clean)
)

print(
    "Registros preparados para análisis:",
    len(sales_analysis)
)

print(
    "Registros excluidos:",
    len(sales_clean) - len(sales_analysis)
)

In [0]:
#Inventory: limpieza mínima
## Para parte y fecha
inventory_clean["fl_RegistroValido"] = (
    inventory_clean["fl_ParteValida"]
    &
    inventory_clean["fl_FechaValida"]
)

inventory_analysis = (
    inventory_clean[
        inventory_clean["fl_RegistroValido"]
    ]
    .copy()
)

print(
    "Inventory original:",
    len(inventory_clean)
)

print(
    "Inventory preparado:",
    len(inventory_analysis)
)

#Resumen de Cleaning Rules


In [0]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "Sales",
        "Inventory"
    ],
    
    "Original Rows": [
        len(sales_clean),
        len(inventory_clean)
    ],
    
    "Prepared Rows": [
        len(sales_analysis),
        len(inventory_analysis)
    ],
    
    "Excluded Rows": [
        len(sales_clean) - len(sales_analysis),
        len(inventory_clean) - len(inventory_analysis)
    ]
})

display(cleaning_summary)

#Resultado de las reglas de Limpieza
Las reglas de limpieza fueron definidas a partir de los hallazgos identifciados durante el data profiling.

Los registro con valores potencialmente inválidos no fueron eliminados directamente del dataset original. En su lugar, se utilizaron banderas de calidad para identificar los observables que requeiren revisión.

Para las ventas se validaron:

- Número de parte
- Cantidad vendida
- Precio de venta
- Fecha de la transacción

Posteriormetne se generó un dataset idnependinete con los registros que cumplieron las condiciones mínimas para continuar con el análisis.

Este enfoque permite conservar los registros originales y mantener trazabilidad sobre las razones por las cuales una observación no fue incorporada al dataset analítico.

#Integration Validation

**Objetivo**, Demostrar que cd_NumParte permíte relacionar Sales e Inventory antes de ejecutar merge

In [0]:
#Crear los conjuntos de números de parte
sales_parts = set( #set busca elementos únicos
    sales_analysis["cd_NumParte"].dropna()
)

inventory_parts = set(
    inventory_analysis["cd_NumParte"].dropna()
)

print("Partes únicas en Sales:", len(sales_parts))
print("Partes únicas en Inventory:", len(inventory_parts))

In [0]:
#Encontrando partes comunes
common_parts = sales_parts & inventory_parts

print(
    "Partes presentes en ambos datasets:",
    len(common_parts)
)

In [0]:
#Identificar partes que existen solo en Sales
sales_only = sales_parts - inventory_parts

print(
    "Partes presentes solo en Sales:",
    len(sales_only)
)

In [0]:
#Partes solo en inventory
inventory_only = inventory_parts - sales_parts

print(
    "Partes presentes solo en Inventory:",
    len(inventory_only)
)

print(
    list(inventory_only)[:10]
)

In [0]:
#Calcular porcentaje de correspondencia
if len(sales_parts) > 0:

    match_percentage = (
        len(common_parts)
        / len(sales_parts)
        * 100
    )

else:
    match_percentage = 0

print(
    f"Correspondencia Sales → Inventory: "
    f"{match_percentage:.2f}%"
)

La fórmula de correspondencia es:

Match% = (Partes comunes / Partes únicas en Sales) × 100

In [0]:
#Creando un resumen de integración
integration_summary = pd.DataFrame({
    "Metric": [
        "Unique parts in Sales",
        "Unique parts in Inventory",
        "Parts in both datasets",
        "Parts only in Sales",
        "Parts only in Inventory",
        "Sales → Inventory match %"
    ],

    "Value": [
        len(sales_parts),
        len(inventory_parts),
        len(common_parts),
        len(sales_only),
        len(inventory_only),
        round(match_percentage, 2)
    ]
})

display(integration_summary)

#Resultado de la validación de integración
La variable cd_NumParte fue seleccionada como campo común para relacionar los datasets de Sales e Inventory.

Antes de ejecutar la integración se compararon los números de parte únicos presentes en ambos conjuntos de datos.

Se encontraron:

953 números de parte únicos en Sales.

1000 números de parte únicos en Inventory.

953 números de parte presentes en ambos datasets.

0 números de parte presentes únicamente en Sales.

47 números de parte presentes únicamente en Inventory.

La correspondencia de números de parte de Sales hacia Inventory fue de 100%.

Los números de parte que no encontraron correspondencia no fueron corregidos o eliminados automáticamente. Estos casos se consideran diferencias que requieren revisión de catálogo, temporalidad o reglas de negocio.

Con base en esta validación se considera que cd_NumParte puede utilizarse como campo de relación para construir el dataset analítico del prototipo.

#Merge and Export



In [0]:
#Revisar granularidad antes del merge
print(
    "Filas Sales preparadas:",
    len(sales_analysis)
)

print(
    "Filas Inventory preparadas:",
    len(inventory_analysis)
)

print(
    "Partes únicas en Inventory:",
    inventory_analysis["cd_NumParte"].nunique()
)

In [0]:
#Verficiar duplicados de parte en Inventory
inventory_part_duplicates = (
    inventory_analysis[
        inventory_analysis.duplicated(
            subset=["cd_NumParte"],
            keep=False
        )
    ]
)

print(
    "Registros de Inventory con "
    "cd_NumParte duplicado:",
    len(inventory_part_duplicates)
)

In [0]:
#Seleccionar columnas de Inventory

#Comprobar nombres con inventory_analysis.columns.tolist()

inventory_selected = inventory_analysis[
    [
        "cd_NumParte",
        "tx_Descripcion",
        "ct_Existencia",
        "ct_Reservado",
        "nu_PuntoReorden",
        "im_ValorInventario",
        "ct_VentasUltimos12Meses"
    ]
].copy()

In [0]:
#Realizar Merge
sellout_analytical = sales_analysis.merge(
    inventory_selected,
    on="cd_NumParte", #indica el campo que relaciona ambos datasets
    how="left", #Conserva todas las ventas válidas, no inner
    validate="many_to_one",
    indicator=True
)

In [0]:
#Validar el Merge
print(
    "Filas antes del merge:",
    len(sales_analysis)
)

print(
    "Filas después del merge:",
    len(sellout_analytical)
)


In [0]:
#Revisar Correspondencias
display(
    sellout_analytical["_merge"]
    .value_counts()
)

#En porcentaje
merge_match_percentage = (
    sellout_analytical["_merge"]
    .eq("both")
    .mean()
    * 100
)

print(
    f"Correspondencia del merge: "
    f"{merge_match_percentage:.2f}%"
)


In [0]:
#Visaulización del resultado
display(
    sellout_analytical.head(10)
)

#Eliminar columna técnica "_merge"
sellout_analytical = (
    sellout_analytical
    .drop(columns=["_merge"])
)

In [0]:
#Exportar el dataset
output_file = (
    PROCESSED_PATH
    / "sellout_analytical_dataset.csv"
)

sellout_analytical.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(
    "Dataset exportado correctamente:"
)

print(output_file)

In [0]:
#Validar que realmente existe
print(
    "¿El archivo existe?",
    output_file.exists()
)

print(
    "Filas exportadas:",
    len(sellout_analytical)
)

print(
    "Columnas exportadas:",
    sellout_analytical.shape[1]
)

#Resultado de Data Preparation

Como resultado del proceso de preparación se generó un dataset analítico integrando la información preparada de ventas con las variables relevantes del inventario mediante cd_NumParte.

Antes de realizar la integración se validó la correspondencia entre los números de parte de ambos datasets.

El proceso utilizó un left join para conservar todas las transacciones de venta válidas, incluso cuando algún producto no encontrara correspondencia en inventario.

También se utilizó una validación many-to-one para controlar que múltiples transacciones de venta pudieran corresponder a un único registro de inventario por número de parte, evitando la duplicación accidental de transacciones durante la integración.

Finalmente, el resultado fue exportado como:

data/processed/sellout_analytical_dataset.csv

Este archivo será utilizado como entrada para el análisis exploratorio y la generación de indicadores del siguiente notebook.